### Make a temporary space in /flash

In [1]:
import os, random, string

In [2]:
cwd = os.path.basename(os.getcwd())
tmpdir = os.path.join("/flash/ReiterU/tmp/", 'tomo_' + ''.join(random.choices(string.ascii_uppercase, k=12)))
print(tmpdir)
if not os.path.exists(tmpdir):
    os.makedirs(tmpdir)

/flash/ReiterU/tmp/tomo_IIMQIFTHAEAO


### PARAMETERS

In [3]:
#### path to relevant files
rootdir = "/bucket/ReiterU/OLdev/experiments/20250520_squid_cal520"
recname = "rec7"
savedir = "/bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec7"

templatefile = os.path.join(rootdir, "2P", recname, "run2/mcorr_nonrigid/QC/QC_nonrigid/template.tif")
roifile = os.path.join(rootdir, "2P", recname, "ROI_transformed_corrected.zip")
dfffile = os.path.join(rootdir, "2P", recname, "dff_ROI_transformed_corrected/DFF.hdf5")
syncfile = os.path.join(rootdir, "sync", f"{recname}.tdms")

QI_THRESH = 0.25

### Import stuff

In [4]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import signal
import h5py
import subprocess

In [5]:
import sys
sys.path.append("../../../")
from calfuncs.io import read_sync_ttl
from calfuncs.full_field_step import get_ett2, plot_chirp_eta, compute_qi

In [6]:
from matplotlib import font_manager
import matplotlib as mpl

font_manager.fontManager.addfont('/bucket/ReiterU/common/fonts/arial.ttf')
mpl.rcParams.update({
    'font.size': 7,
    'font.family': 'sans-serif',
    'font.sans-serif': "Arial",
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'xtick.major.size': 2,
    'xtick.minor.size': 2,
    'xtick.minor.width': 0.5,
    'ytick.major.width': 0.5,
    'ytick.minor.width': 0.5,
    'ytick.major.size': 2,
    'ytick.minor.size': 2,
})

### Load TDMS file

In [7]:
sync_ttl = read_sync_ttl(syncfile)
twop_ttl = sync_ttl["twop_ttl"]
twop_re = sync_ttl["twop_re"]
vis_re = sync_ttl["vis_re"]
print(twop_ttl.shape, twop_re.shape, vis_re.shape)

(719000,) (10943,) (60,)


### Load DF/F

In [8]:
with h5py.File(dfffile, "r") as hf:
    F = hf["F"][:]
    dff = hf["dff"][:]
print(F.shape) # F is sampled at 1 kH

(176, 719000)


### Chirp

In [9]:
E = []
for nid in np.arange(0, F.shape[0]):
    ett1 = get_ett2(F, nid, vis_re[0::6], window=[-2000, 19000], baseline_window=[-4000, 70000], subsampling=50)
    ett1["session"] = 0
    ett2 = get_ett2(F, nid, vis_re[2::6], window=[-2000, 19000], baseline_window=[-27000, 43000], subsampling=50)
    ett2["session"] = 1
    ett3 = get_ett2(F, nid, vis_re[4::6], window=[-2000, 19000], baseline_window=[-51000, 19000], subsampling=50)
    ett3["session"] = 2
    ett = pd.concat([ett1, ett2, ett3])
    ett["neuron"] = nid
    ett = ett.set_index(["neuron", "session", "trial", "time"])
    E.append(ett)
E = pd.concat(E)

In [ ]:
dt = 1e-2
stimulus = np.zeros(int(21.0/dt), dtype=np.float32)
stimulus[0:int(2.0/dt)] = 0
stimulus[int(2.0/dt):int(6.0/dt)] = 1
stimulus[int(6.0/dt):int(10.0/dt)] = 0
stimulus[int(10.0/dt):int(12.0/dt)] = 0.5
stimulus[int(12.0/dt):int(20.0/dt)] = 0.5 + 0.5 * signal.chirp(np.linspace(0, 8, int(20.0/dt)-int(12.0/dt)), f0=0.5, f1=10, t1=8, method='logarithmic', phi=90)
stimulus[int(20.0/dt):int(21.0/dt)] = 0

for nid in np.arange(0, dff.shape[0]):
    qi = compute_qi(E, nid)
    print("Neuron:", nid, "QI:", qi)
    savename = os.path.join(tmpdir, f"ETA_{nid:03d}.png")
    plot_chirp_eta(E, nid, stimulus, vmin=-2, vmax=3,  cmap_name="bone_r", figsize=(5,2.0), show_plot=True, figsavename=savename)

### Compute trial-average

In [11]:
R = []
neurons = E.index.get_level_values("neuron").unique()

for n in neurons:
    M = []
    df2 = E.xs(n, level="neuron")
    for i in range(3):
        df3 = df2.xs(i, level="session")
        mat = df3.reset_index().pivot(index="trial", columns="time", values="y").to_numpy()
        M.append(mat)
    M = np.hstack(M)
    M = M[~np.all(np.isnan(M), axis=1)] # remove trials with NaN values
    qi = compute_qi(E, n)
    y_mean = np.nanmean(M, axis=0)
    row = {
        "recording": 0,
        "neuron": n,
        "qi": qi,
    }
    for j, value in enumerate(y_mean):
        row[f"time_{j}"] = value

    R.append(row)

R = pd.DataFrame(R)

In [12]:
display(R)

,recording,neuron,qi,time_0,time_1,time_2,time_3,time_4,time_5,time_6,...,time_1250,time_1251,time_1252,time_1253,time_1254,time_1255,time_1256,time_1257,time_1258,time_1259
0,0,0,0.368475,0.224756,-0.184214,-0.809338,-0.340909,-0.041954,-0.469127,-1.026149,...,-0.093515,-0.277559,0.229259,0.603949,0.786535,0.347130,0.534281,0.752546,0.668428,0.396467
1,0,1,0.528242,-0.355094,-0.289736,-0.674708,-0.443536,-0.517367,-0.502167,-0.231627,...,-0.328224,-0.445505,-0.505221,-0.599895,-0.619137,-0.513210,-0.254238,-0.366018,-0.612294,-0.615747
2,0,2,0.487497,-0.664833,-0.856951,-1.244459,-0.985804,-0.690117,-0.919448,-0.707369,...,-0.410046,-0.323324,-0.688260,-0.971332,-0.663304,-0.534722,-0.581302,-0.652380,-0.860242,-0.927557
3,0,3,0.440571,0.794233,1.016523,0.830395,0.965279,0.520166,0.372067,0.574904,...,-0.532777,-0.436767,-0.836808,-1.038972,-0.848796,-0.890048,-0.788418,-0.883104,-0.950754,-0.896130
4,0,4,0.205138,0.971765,0.526732,0.083055,0.421457,0.990954,1.004779,1.145541,...,-0.349661,0.052732,-0.261955,-0.658546,-0.304092,-0.116118,-0.653354,-0.652874,-0.432448,-0.187054
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,0,171,0.375245,0.668144,0.587904,0.722463,0.697383,0.466873,0.620209,0.385886,...,-1.112848,-1.178150,-0.629541,-0.498064,-0.732518,-0.876765,-0.709861,-0.688761,-0.790434,-0.916833
172,0,172,0.735705,0.251487,0.214301,-0.023305,-0.109950,-0.013511,0.050981,-0.096993,...,-0.903824,-0.953192,-1.060130,-0.985129,-0.898782,-0.943395,-1.075657,-0.945765,-0.936868,-1.049275
173,0,173,0.185348,-0.323821,-0.485542,-0.463862,-0.007661,0.068520,0.001071,-0.109860,...,0.082140,0.149639,-0.411791,-0.192491,-0.081879,-0.261517,-0.510987,-0.393533,-0.166588,-0.009749
174,0,174,0.159433,0.195716,0.499384,1.121417,1.191989,0.868039,0.942507,0.864790,...,-0.327953,0.145050,-0.459502,-0.175033,-0.139346,-0.182896,-0.165012,-0.008556,-0.069733,-0.117928


In [13]:
with pd.HDFStore(os.path.join(tmpdir, "response_table.hdf5"), mode='w') as store:
    store.put('E', E)
    store.put('responses', R)

### Copy data to bucket

In [14]:
cmd1 = f'ssh deigo "mkdir -p {savedir}"'
cmd2 = f'ssh deigo "cp -r {tmpdir}/* {savedir}"'
print(cmd1)
print(cmd2)

subprocess.run(cmd1, shell=True, check=True)
subprocess.run(cmd2, shell=True, check=True)

ssh deigo "mkdir -p /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec7"
ssh deigo "cp -r /flash/ReiterU/tmp/tomo_IIMQIFTHAEAO/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec7"


Offending key for IP in /home/t/tomoyuki-mano1/.ssh/known_hosts:1
Matching host key in /home/t/tomoyuki-mano1/.ssh/known_hosts:6


CompletedProcess(args='ssh deigo "cp -r /flash/ReiterU/tmp/tomo_IIMQIFTHAEAO/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250520_rec7"', returncode=0)